<a href="https://colab.research.google.com/github/IvanBaroni/projects-in-data/blob/main/Titanic_Dataset_Exploration_WIP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd

# Construct the download URL for the Google Sheet
sheet_id = '1I0L6bt8ZF_vSeHNWHySBlsAkT1DNxJhWc3FmDaZTbzo'
url = f'https://docs.google.com/spreadsheets/d/{sheet_id}/export?format=csv'

# Read the data from the URL into a pandas DataFrame
df = pd.read_csv(url)

# Display the first 5 rows of the DataFrame
display(df.head())

,pclass,survived,name,sex,age,sibsp,parch,ticket,fare,cabin,embarked,boat,body,home.dest
0,3,0,"Abbing, Mr. Anthony",male,42.0,0,0,C.A. 5547,7.55,NaN,S,NaN,NaN,NaN
1,3,0,"Abbott, Master. Eugene Joseph",male,13.0,0,2,C.A. 2673,20.25,NaN,S,NaN,NaN,"East Providence, RI"
2,3,0,"Abbott, Mr. Rossmore Edward",male,16.0,1,1,C.A. 2673,20.25,NaN,S,NaN,190.0,"East Providence, RI"
3,3,1,"Abbott, Mrs. Stanton (Rosa Hunt)",female,35.0,1,1,C.A. 2673,20.25,NaN,S,A,NaN,"East Providence, RI"
4,3,1,"Abelseth, Miss. Karen Marie",female,16.0,0,0,348125,7.65,NaN,S,16,NaN,"Norway Los Angeles, CA"


In [2]:
df_summary = pd.DataFrame({
    'Null Count': df.isnull().sum(),
    'Unique Count': df.nunique(),
    'Type': df.dtypes
    })

display(df_summary)

print(f"The dataframe has {df.shape[0]:,} rows and {df.shape[1]:,} columns.")

,Null Count,Unique Count,Type
pclass,0,3,int64
survived,0,2,int64
name,0,1307,object
sex,0,2,object
age,263,98,float64
sibsp,0,7,int64
parch,0,8,int64
ticket,0,929,object
fare,1,281,float64
cabin,1014,186,object


The dataframe has 1,309 rows and 14 columns.


In [3]:
# ydata profiling

'''
!pip install ydata-profiling > /dev/null
from ydata_profiling import ProfileReport # Used for generating profile reports from a pandas DataFrame

profile = ProfileReport(df, title="Profiling Report")
profile.to_notebook_iframe()
'''

'\n!pip install ydata-profiling > /dev/null\nfrom ydata_profiling import ProfileReport # Used for generating profile reports from a pandas DataFrame\n\nprofile = ProfileReport(df, title="Profiling Report")\nprofile.to_notebook_iframe()\n'

## Selecting only columns of interest

In [4]:
new_df = df[['name', 'pclass', 'sex', 'age', 'sibsp', 'parch', 'fare', 'embarked', 'survived']].copy()

new_df = new_df.rename(columns={
    'name': 'Passenger_Name',
    'pclass': 'Passenger_Class',
    'sex': 'Sex',
    'age': 'Age',
    'sibsp': 'Sibligns_or_Spouse',
    'parch': 'Parents_or_Children',
    'fare': 'Fare',
    'embarked': 'Embarked_Port',
    'survived': 'Survived'
})

display(new_df.head())

,Passenger_Name,Passenger_Class,Sex,Age,Sibligns_or_Spouse,Parents_or_Children,Fare,Embarked_Port,Survived
0,"Abbing, Mr. Anthony",3,male,42.0,0,0,7.55,S,0
1,"Abbott, Master. Eugene Joseph",3,male,13.0,0,2,20.25,S,0
2,"Abbott, Mr. Rossmore Edward",3,male,16.0,1,1,20.25,S,0
3,"Abbott, Mrs. Stanton (Rosa Hunt)",3,female,35.0,1,1,20.25,S,1
4,"Abelseth, Miss. Karen Marie",3,female,16.0,0,0,7.65,S,1


## Feature Engeneering

### Feature: Family Size

In [5]:
# Create Family_Size
new_df['Family_Size'] = new_df['Sibligns_or_Spouse'] + new_df['Parents_or_Children'] + 1

# Insert the 'Family_Size' column at the seventh position (index 6)
new_df.insert(6, 'Family_Size', new_df.pop('Family_Size'))

In [6]:
import plotly.express as px
import pandas as pd

# Calculate the counts for each Family_Size and sort them
family_size_counts = new_df['Family_Size'].value_counts().sort_index().reset_index()
family_size_counts.columns = ['Family_Size', 'Count']

fig = px.bar(family_size_counts, x='Family_Size', y='Count', title='Distribution of Passengers by Family Size', color_discrete_sequence=['blue'],
             width=1400, height=500) # Set chart size

# Define tick values and labels explicitly from the sorted counts
tick_vals = family_size_counts['Family_Size'].tolist()
tick_text = [str(val) for val in tick_vals]

fig.update_layout(
    xaxis_title='Family Size',
    yaxis_title='Number of Passengers',
    yaxis={'visible': False, 'showticklabels': False, 'showgrid': True, 'gridcolor': 'white'}, # Show white grid for y-axis
    xaxis=dict(
        tickmode = 'array',
        tickvals = tick_vals,
        ticktext = tick_text,
        tickfont = dict(weight = 'bold'),
        showgrid=True, gridcolor='white' # Show white grid for x-axis
    ),
    plot_bgcolor='white', # Set plot background to white (effectively removing the light grey)
    paper_bgcolor='white', # Set paper background to white
    shapes=[
        dict(
            type='line',
            xref='paper', x0=0, x1=1,
            yref='paper', y0=0, y1=0,
            line=dict(color='grey', width=2)
        )
    ]
)

# Add text annotations on top of the bars and in blue font
fig.update_traces(texttemplate='%{y}', textposition='outside', textfont=dict(color='blue', size=12, weight="bold"))


fig.show()

### Feature: Is Alone?

In [7]:
new_df['Is_Alone'] = new_df['Family_Size'].apply(lambda x: 'Yes' if x == 1 else 'No')

# Insert the 'Family_Size' column at the seventh position (index 6)
new_df.insert(7, 'Is_Alone', new_df.pop('Is_Alone'))

In [8]:
import plotly.express as px

is_alone_counts = new_df['Is_Alone'].value_counts().reset_index()
is_alone_counts.columns = ['Is_Alone', 'Count']

fig = px.pie(is_alone_counts, values='Count', names='Is_Alone', title='') # Set title to empty string to hide it
fig.update_traces(textposition='inside', textinfo='percent+value+label', hole=.55, # Add hole for donut chart and include value in text
                  textfont=dict(weight='bold', size=14), # Make labels bold and increase size
                  marker=dict(line=dict(color='white', width=2))) # Add white border to slices

# Add text annotation in the center
fig.update_layout(
    annotations=[dict(text='Is Alone?', x=0.5, y=0.5, font_size=22, showarrow=False, font=dict(weight='bold'))] # Make central text bold
)

# Set colors for slices
fig.update_traces(marker=dict(colors=['black', 'lightgrey']))


fig.show()

### Feature: Age Group

In [9]:
# Define the age bins and labels with more granular adult tiers
bins = [0, 4, 12, 18, 30, 45, 60, new_df['Age'].max()]
age_labels = ['Baby', 'Child', 'Teen', 'Young Adult', 'Midlife Adult', 'Senior Adult', 'Senior']

# Create the Age_Group column, handling missing values
new_df['Age_Group'] = pd.cut(new_df['Age'], bins=bins, labels=age_labels, right=False)
new_df['Age_Group'] = new_df['Age_Group'].cat.add_categories('Unknown')
new_df['Age_Group'] = new_df['Age_Group'].fillna('Unknown')

new_df.insert(4, 'Age_Group', new_df.pop('Age_Group'))

In [10]:
import plotly.express as px

# Define the age labels in the desired order
age_labels = ['Baby', 'Child', 'Teen', 'Young Adult', 'Midlife Adult', 'Senior Adult', 'Senior', 'Unknown']

fig = px.histogram(new_df, x='Age_Group', title='Distribution of Passengers by Age Group',
                   category_orders={'Age_Group': age_labels}, width=1500, height=500) # Set chart size

# Update layout to order bars and add titles
fig.update_layout(
    xaxis_title='Age Group',
    yaxis_title='Number of Passengers',
    yaxis={'visible': False, 'showticklabels': False},
    xaxis={'tickfont': {'weight': 'bold', 'size': 14}}, # Make x-axis labels bold and increase size
    plot_bgcolor='white', # Remove plot background
    paper_bgcolor='white', # Remove paper background
    shapes=[
        dict(
            type='line',
            xref='paper', x0=0, x1=1,
            yref='paper', y0=0, y1=0,
            line=dict(color='grey', width=2)
        )
    ]
)

# Create a list of colors based on the age groups
colors = ['green'] * len(age_labels)
unknown_index = age_labels.index('Unknown')
colors[unknown_index] = '#a2a6a3' # Set color for Unknown bar

# Set the colors of the bars
fig.update_traces(marker_color=colors)

# Add text annotations inside the bars and in white font
fig.update_traces(texttemplate='%{y}', textposition='inside', textfont=dict(color='white', size=20, family="Arial", weight="bold"))


fig.show()

### Feature: Title

In [11]:
import pandas as pd # Ensure pandas is imported if not already

# Extract title using pandas string operations
new_df['Title'] = new_df['Passenger_Name'].str.extract(' ([A-Za-z]+)\.', expand=False)

# Reorder columns to place 'Title' after 'Passenger_Name'
cols = new_df.columns.tolist()
cols.insert(cols.index('Passenger_Name') + 1, cols.pop(cols.index('Title')))
new_df = new_df[cols]

In [12]:
import plotly.express as px

title_counts = new_df['Title'].value_counts().reset_index()
title_counts.columns = ['Title', 'Count']

fig = px.bar(title_counts, x='Title', y='Count', title='Distribution of Passengers by Title',
             color_discrete_sequence=['red'], width=1800, height=600) # Set bar color and chart size

fig.update_layout(
    xaxis_title='Title',
    yaxis_title='Number of Passengers',
    yaxis={'visible': False, 'showticklabels': False}, # Get rid of y-axis
    xaxis={'tickfont': {'weight': 'bold'}}, # Make x-axis labels bold
    plot_bgcolor='white', # Remove plot background
    paper_bgcolor='white', # Remove paper background
    shapes=[
        dict(
            type='line',
            xref='paper', x0=0, x1=1,
            yref='paper', y0=0, y1=0, # Position the line at the top of the plot area
            line=dict(color='grey', width=2)
        )
    ]
)

# Put the numbers on top of each bar
fig.update_traces(texttemplate='%{y}', textposition='outside', textfont=dict(color='red', size=12, weight="bold")) # Make text bold and red


fig.show()

In [13]:
import plotly.express as px

# Define a color map for passenger classes
color_map = {1: '#fe218b', 2: '#fed700', 3: '#21b0fe'}

# Define the order of titles based on their counts
title_order = title_counts['Title'].tolist()

# Create a stacked bar chart of Title distribution by Passenger_Class
fig = px.histogram(new_df, x='Title', color='Passenger_Class',
                   title='Distribution of Titles by Passenger Class',
                   width=1600, height=600,
                   color_discrete_map=color_map,
                   category_orders={"Title": title_order, "Passenger_Class": [1, 2, 3]}) # Use category_orders for both axes


fig.update_layout(
    xaxis_title='Title',
    yaxis_title='Number of Passengers (Log Scale)', # Update y-axis title for clarity
    yaxis={'visible': True, 'showticklabels': False, 'type': 'log'}, # Set y-axis to log scale and make visible
    xaxis={'tickfont': {'weight': 'bold'}},
    plot_bgcolor='white',
    paper_bgcolor='white',
    shapes=[
        dict(
            type='line',
            xref='paper', x0=0, x1=1,
            yref='paper', y0=0, y1=0,
            line=dict(color='grey', width=2)
        )
    ],
    barmode='group' # Change barmode to 'group' for side-by-side bars
)

# Add text annotations on top of the bars and make text bold and match bar color
for i, trace in enumerate(fig.data):
    # The name of the trace is the Passenger_Class value (1, 2, or 3)
    pclass = int(trace.name)
    color = color_map.get(pclass) # Get the color from the color_map
    if color:
        fig.update_traces(selector=dict(name=str(pclass)), # Select trace by string name
                          texttemplate='%{y}', textposition='outside',
                          textfont=dict(weight='bold', color=color))

# Update legend labels
new_legend_names = {1: 'First Class', 2: 'Second Class', 3: 'Third Class'}
fig.for_each_trace(lambda t: t.update(name = new_legend_names[int(t.name)]))


fig.show()

In [14]:
# Set pandas display option to show more rows
pd.set_option('display.max_rows', None) # Set to None to display all rows, or a large integer

display(new_df.head(15)) # Displaying first 100 rows as requested previously


,Passenger_Name,Title,Passenger_Class,Sex,Age,Age_Group,Sibligns_or_Spouse,Parents_or_Children,Family_Size,Is_Alone,Fare,Embarked_Port,Survived
0,"Abbing, Mr. Anthony",Mr,3,male,42.00,Midlife Adult,0,0,1,Yes,7.5500,S,0
1,"Abbott, Master. Eugene Joseph",Master,3,male,13.00,Teen,0,2,3,No,20.2500,S,0
2,"Abbott, Mr. Rossmore Edward",Mr,3,male,16.00,Teen,1,1,3,No,20.2500,S,0
3,"Abbott, Mrs. Stanton (Rosa Hunt)",Mrs,3,female,35.00,Midlife Adult,1,1,3,No,20.2500,S,1
4,"Abelseth, Miss. Karen Marie",Miss,3,female,16.00,Teen,0,0,1,Yes,7.6500,S,1
5,"Abelseth, Mr. Olaus Jorgensen",Mr,3,male,25.00,Young Adult,0,0,1,Yes,7.6500,S,1
6,"Abelson, Mr. Samuel",Mr,2,male,30.00,Midlife Adult,1,0,2,No,24.0000,C,0
7,"Abelson, Mrs. Samuel (Hannah Wizosky)",Mrs,2,female,28.00,Young Adult,1,0,2,No,24.0000,C,1
8,"Abrahamsson, Mr. Abraham August Johannes",Mr,3,male,20.00,Young Adult,0,0,1,Yes,7.9250,S,1
9,"Abrahim, Mrs. Joseph (Sophie Halaut Easu)",Mrs,3,female,18.00,Young Adult,0,0,1,Yes,7.2292,C,1


## Exploring Survival by Variables

### Survival by Sex

In [15]:
import plotly.express as px
from plotly.subplots import make_subplots
import plotly.graph_objects as go

# Create subplots: 1 row, 3 columns
fig = make_subplots(rows=1, cols=3, specs=[[{'type':'domain'}, {'type':'domain'}, {'type':'domain'}]]) # Removed subplot_titles

# Define colors for male and female
color_map = {'male': '#456990', 'female': '#ef767a'}
# Define the desired order of sexes with new labels
sex_order = ['female', 'male']
new_sex_labels = {'male': '♂ Male', 'female': '♀ Female'}
new_sex_order = ['♀ Female', '♂ Male']


# 1) Total Male vs Female
sex_counts_total = new_df['Sex'].value_counts().reindex(sex_order).reset_index()
sex_counts_total.columns = ['Sex', 'Count']
sex_counts_total['Sex'] = sex_counts_total['Sex'].map(new_sex_labels) # Map to new labels

fig.add_trace(go.Pie(labels=sex_counts_total['Sex'], values=sex_counts_total['Count'], name='Total',
                     marker=dict(colors=[color_map[sex] for sex in sex_counts_total['Sex'].map({v: k for k, v in new_sex_labels.items()})])), # Apply colors based on original sex
              1, 1)

# Filter for survived passengers
survived_df = new_df[new_df['Survived'] == 1]

# 2) Total Male vs Female for Survived = 1
sex_counts_survived = survived_df['Sex'].value_counts().reindex(sex_order).reset_index()
sex_counts_survived.columns = ['Sex', 'Count']
sex_counts_survived['Sex'] = sex_counts_survived['Sex'].map(new_sex_labels) # Map to new labels

fig.add_trace(go.Pie(labels=sex_counts_survived['Sex'], values=sex_counts_survived['Count'], name='Survived',
                     marker=dict(colors=[color_map[sex] for sex in sex_counts_survived['Sex'].map({v: k for k, v in new_sex_labels.items()})])), # Apply colors based on original sex
              1, 2)

# Filter for not survived passengers
not_survived_df = new_df[new_df['Survived'] == 0]

# 3) Total Male vs Female for Survived = 0
sex_counts_not_survived = not_survived_df['Sex'].value_counts().reindex(sex_order).reset_index()
sex_counts_not_survived.columns = ['Sex', 'Count']
sex_counts_not_survived['Sex'] = sex_counts_not_survived['Sex'].map(new_sex_labels) # Map to new labels

fig.add_trace(go.Pie(labels=sex_counts_not_survived['Sex'], values=sex_counts_not_survived['Count'], name='Not Survived',
                     marker=dict(colors=[color_map[sex] for sex in sex_counts_not_survived['Sex'].map({v: k for k, v in new_sex_labels.items()})])), # Apply colors based on original sex
              1, 3)

# Update layout for donut charts and text
fig.update_traces(textposition='inside', textinfo='percent+value+label', hole=.55, textfont=dict(weight='bold'), insidetextfont=dict(size=12)) # Convert to donut charts, show text, make hole bigger, and make text bold
fig.update_layout(title_text="Male vs Female Distribution by Survival", showlegend=True, width=1500, height=600) # Add a main title and show legend, set width and height

# Add text annotations in the center of each donut chart
fig.update_layout(
    annotations=[
        dict(text='🚢 Total', x=0.10, y=0.5, font_size=22, showarrow=False, font=dict(weight='bold')), # Position for the first chart and increase font size
        dict(text='🛟 Survived', x=0.5, y=0.5, font_size=22, showarrow=False, font=dict(weight='bold')), # Position for the second chart and increase font size
        dict(text='💀 Died', x=0.89, y=0.5, font_size=22, showarrow=False, font=dict(weight='bold')) # Position for the third chart and increase font size
    ]
)

# Add white border to the slices (re-adding this from a previous request)
fig.update_traces(marker=dict(line=dict(color='white', width=2)))


fig.show()

In [16]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import pandas as pd

# Define colors for male and female
color_male = '#456990'
color_female = '#ef767a'

# Get counts for total, survived, and not survived by sex
sex_counts_total = new_df['Sex'].value_counts().reindex(['male', 'female'])
sex_counts_survived = new_df[new_df['Survived'] == 1]['Sex'].value_counts().reindex(['male', 'female'])
sex_counts_not_survived = new_df[new_df['Survived'] == 0]['Sex'].value_counts().reindex(['male', 'female'])

# Create subplots: 1 row, 3 columns, with shared y-axis
fig = make_subplots(rows=1, cols=3,
                    subplot_titles=('🚢 Total Passengers', '🛟 Survived', '💀 Not Survived'),
                    shared_yaxes=True)

# Add Total count bar chart
fig.add_trace(go.Bar(x=sex_counts_total.index, y=sex_counts_total.values,
                     marker_color=[color_male, color_female],
                     texttemplate='%{y}', textposition='outside',
                     textfont=dict(weight='bold', color=[color_male, color_female], size=14)), 1, 1) # Make text bigger

# Add Survived count bar chart
fig.add_trace(go.Bar(x=sex_counts_survived.index, y=sex_counts_survived.values,
                     marker_color=[color_male, color_female],
                     texttemplate='%{y}', textposition='outside',
                     textfont=dict(weight='bold', color=[color_male, color_female], size=14)), 1, 2) # Make text bigger

# Add Not Survived count bar chart
fig.add_trace(go.Bar(x=sex_counts_not_survived.index, y=sex_counts_not_survived.values,
                     marker_color=[color_male, color_female],
                     texttemplate='%{y}', textposition='outside',
                     textfont=dict(weight='bold', color=[color_male, color_female], size=14)), 1, 3) # Make text bigger

# Update layout
fig.update_layout(title_text="Male vs Female Counts by Survival", showlegend=False, width=1300, height=550,
                  xaxis=dict(tickfont=dict(size=14, weight='bold'))) # Make x-axis labels bigger and bold


fig.show()

### Survival by Age

In [75]:
import plotly.express as px
import plotly.graph_objects as go # Import graph_objects for annotations

# Define the desired order of age groups and the new labels with emojis
age_order = ['Baby', 'Child', 'Teen', 'Young Adult', 'Midlife Adult', 'Senior Adult', 'Senior', 'Unknown']
emoji_age_labels = ['🍼 Baby', '🧸 Child', '🎧 Teen', '☕ Young Adult', '💻 Midlife Adult', '🛋️ Senior Adult', '🌅 Senior', '❔ Unknown']

# Define a color map for the 'Survived' column
color_map = {0: 'black', 1: '#57B9FF'}

fig = px.histogram(new_df, x='Age_Group', color='Survived',
                   title='Distribution of Passengers by Age Group and Survival',
                   category_orders={'Age_Group': age_order},
                   barmode='group',
                   color_discrete_map=color_map,
                   width=1500, height=600) # Set the chart size

fig.update_layout(
    xaxis_title= None,
    yaxis_title='Number of Passengers',
    yaxis={'visible': False, 'showticklabels': False},
    plot_bgcolor='white',
    paper_bgcolor='white',
    xaxis=dict(
        tickmode = 'array',
        tickvals = age_order, # Use the original age group names as values
        ticktext = emoji_age_labels, # Use the emoji labels as the text
        tickfont = dict(size=14, weight='bold') # Make x-axis labels bold and increase size
    ),
    shapes=[
        dict(
            type='line',
            xref='paper', x0=0, x1=1,
            yref='paper', y0=0, y1=0,
            line=dict(color='grey', width=2)
        )
    ]
)

# Add text annotations on top of the bars and style them
fig.update_traces(texttemplate='%{y}', textposition='outside')

# Update text font based on color
for i, trace in enumerate(fig.data):
    survived_status = int(trace.name) # Get the survival status from the trace name (0 or 1)
    color = color_map.get(survived_status) # Get the color from the color_map
    if color:
        fig.update_traces(selector=dict(name=str(survived_status)), # Select trace by string name
                          textfont=dict(color=color, size=14, weight='bold')) # Update text font properties

# Update legend labels for the 'Survived' column
new_legend_names = {0: '💀 No', 1: '🛟 Yes'} # Added emojis to legend
fig.for_each_trace(lambda t: t.update(name = new_legend_names[int(t.name)]))

# Add annotations for survival rate below each Age_Group label
annotations = []
# Make sure survival_rate_table is available from previous cells (cell 8-rP8J6E7CNP)
if 'survival_rate_table' in locals():
    # Iterate through the index of the survival_rate_table to get age group (with emoji) and survival rate
    for age_group_emoji, row in survival_rate_table.iterrows():
        original_age_group = {v: k for k, v in emoji_age_labels_map.items()}.get(age_group_emoji, age_group_emoji) # Get original name for x-position
        survival_rate = row['Survival Rate'] * 100 # Convert to percentage
        annotations.append(dict(
            xref='x', yref='paper', # Use 'x' for x-axis data values, 'paper' for y-axis as fraction of plot height
            x=original_age_group, # Use the original age group name for x-positioning
            y=-0.05, # Position below the x-axis labels (adjust y as needed)
            text=f'Survived: {survival_rate:.0f}%', # Format as percentage with one decimal
            showarrow=False,
            font=dict(size=12,  weight='bold', color='grey'), # Adjust font size and color as needed
            xanchor='center', yanchor='top'
        ))


    fig.update_layout(annotations=annotations, margin=dict(b=100)) # Add annotations to layout and increase bottom margin


fig.show()

In [18]:
import pandas as pd

# Create a crosstab to get survival counts by Age_Group
survival_counts = pd.crosstab(new_df['Age_Group'], new_df['Survived'])

# Rename columns for clarity
survival_counts = survival_counts.rename(columns={0: 'Died', 1: 'Survived'})

# Calculate the total count for each Age_Group
survival_counts['Total'] = survival_counts['Died'] + survival_counts['Survived']

# Calculate the survival rate (Survived / Total)
survival_counts['Survival Rate'] = survival_counts['Survived'] / survival_counts['Total']

# Select and reorder columns for the final table
survival_rate_table = survival_counts[['Total', 'Survived', 'Died', 'Survival Rate']].copy()

# Define the mapping from original age group names to emoji labels
emoji_age_labels_map = {
    'Baby': '🍼 Baby',
    'Child': '🧸 Child',
    'Teen': '🎧 Teen',
    'Young Adult': '☕ Young Adult',
    'Midlife Adult': '💻 Midlife Adult',
    'Senior Adult': '🛋️ Senior Adult',
    'Senior': '🌅 Senior',
    'Unknown': '❔ Unknown'
}

# Map the index (Age_Group) to the emoji labels
survival_rate_table = survival_rate_table.rename(index=emoji_age_labels_map)


# Format the 'Survival Rate' column as percentage with one decimal and make text bold
# Add a light blue bar chart in the background for Survival Rate column with white border
display(survival_rate_table.sort_values(by='Survival Rate', ascending=False).style
        .bar(subset=['Survival Rate'], color='lightblue', props='border: 1px solid white;')
        .format({'Survival Rate': lambda x: f'<b>{x:.1%}</b>'}) # Changed to .1% for one decimal place
        .set_properties(**{'text-align': 'left'}, axis=0) # Align index to the left
        .set_properties(**{'text-align': 'center'}, axis=1)) # Align columns to the center

Survived,Total,Survived,Died,Survival Rate
Age_Group,,,,
🍼 Baby,41,26,15,63.4%
🧸 Child,50,25,25,50.0%
🎧 Teen,63,30,33,47.6%
🛋️ Senior Adult,136,64,72,47.1%
💻 Midlife Adult,301,118,183,39.2%
☕ Young Adult,415,152,263,36.6%
🌅 Senior,39,11,28,28.2%
❔ Unknown,264,74,190,28.0%


### Suvival by Family Size

In [85]:
import plotly.express as px
import plotly.graph_objects as go # Import graph_objects for annotations

# Define a color map for the 'Survived' column
color_map = {0: 'black', 1: '#57B9FF'}

# Get the unique Family_Size values and sort them for the x-axis order
family_size_order = sorted(new_df['Family_Size'].unique())

fig = px.histogram(new_df, x='Family_Size', color='Survived',
                   title='Distribution of Passengers by Family Size and Survival',
                   category_orders={'Family_Size': family_size_order}, # Set the order for Family_Size
                   barmode='group',
                   color_discrete_map=color_map,
                   width=1400, height=600) # Set the chart size

fig.update_layout(
    xaxis_title=None, # Update x-axis title
    yaxis_title='Number of Passengers',
    yaxis={'visible': False, 'showticklabels': False},
    plot_bgcolor='white',
    paper_bgcolor='white',
    xaxis=dict(
        tickmode = 'array',
        tickvals = family_size_order, # Use the Family_Size values as tickvals
        ticktext = [str(size) for size in family_size_order], # Use string representation of sizes as ticktext
        tickfont = dict(size=14, weight='bold') # Make x-axis labels bold and increase size
    ),
    shapes=[
        dict(
            type='line',
            xref='paper', x0=0, x1=1,
            yref='paper', y0=0, y1=0,
            line=dict(color='grey', width=2)
        )
    ]
)

# Add text annotations on top of the bars and style them
fig.update_traces(texttemplate='%{y}', textposition='outside')

# Update text font based on color
for i, trace in enumerate(fig.data):
    survived_status = int(trace.name) # Get the survival status from the trace name (0 or 1)
    color = color_map.get(survived_status) # Get the color from the color_map
    if color:
        fig.update_traces(selector=dict(name=str(survived_status)), # Select trace by string name
                          textfont=dict(color=color, size=14, weight='bold')) # Update text font properties

# Update legend labels for the 'Survived' column
new_legend_names = {0: '💀 No', 1: '🛟 Yes'} # Added emojis to legend
fig.for_each_trace(lambda t: t.update(name = new_legend_names[int(t.name)]))

# Add annotations for survival rate below each Family_Size label
annotations = []
# Make sure survival_rate_table_family_size is available from previous cells (cell 8jDhv1V1qff8)
if 'survival_rate_table_family_size' in locals():
    # Iterate through the index of the survival_rate_table_family_size to get Family_Size and survival rate
    for family_size, row in survival_rate_table_family_size.iterrows():
        survival_rate = row['Survival Rate'] * 100 # Convert to percentage
        annotations.append(dict(
            xref='x', yref='paper', # Use 'x' for x-axis data values, 'paper' for y-axis as fraction of plot height
            x=family_size, y=-0.07, # Position below the x-axis labels (adjust y as needed)
            text=f'Surv: {survival_rate:.0f}%', # Format as percentage with one decimal
            showarrow=False,
            font=dict(size=10, color='grey'), # Adjust font size and color as needed
            xanchor='center', yanchor='top'
        ))


    fig.update_layout(annotations=annotations, margin=dict(b=100)) # Add annotations to layout and increase bottom margin


fig.show()

In [40]:
import pandas as pd

# Create a crosstab to get survival counts by Family_Size
survival_counts_family_size = pd.crosstab(new_df['Family_Size'], new_df['Survived'])

# Rename columns for clarity
survival_counts_family_size = survival_counts_family_size.rename(columns={0: 'Died', 1: 'Survived'})

# Calculate the total count for each Family_Size
survival_counts_family_size['Total'] = survival_counts_family_size['Died'] + survival_counts_family_size['Survived']

# Calculate the survival rate (Survived / Total)
survival_counts_family_size['Survival Rate'] = survival_counts_family_size['Survived'] / survival_counts_family_size['Total']

# Select and reorder columns for the final table
survival_rate_table_family_size = survival_counts_family_size[['Total', 'Survived', 'Died', 'Survival Rate']].copy()

# Format the 'Survival Rate' column as percentage with one decimal and make text bold
# Add a light blue bar chart in the background for Survival Rate column with white border
styled_table = survival_rate_table_family_size.sort_values(by='Family_Size', ascending=True).style # Sort by Family_Size ascending
styled_table = styled_table.bar(subset=['Survival Rate'], color='lightblue', props='border: 1px solid white;')
styled_table = styled_table.format({'Survival Rate': lambda x: f'<b>{x:.1%}</b>'})

# Apply text alignment styling to specific columns by name
styled_table = styled_table.set_properties(subset=['Total', 'Survived', 'Died', 'Survival Rate'], **{'text-align': 'center'})

display(styled_table)

Survived,Total,Survived,Died,Survival Rate
Family_Size,,,,
1,790,239,551,30.3%
2,235,126,109,53.6%
3,159,90,69,56.6%
4,43,30,13,69.8%
5,22,6,16,27.3%
6,25,5,20,20.0%
7,16,4,12,25.0%
8,8,0,8,0.0%
11,11,0,11,0.0%


### Survival by Class

In [62]:
import plotly.express as px
import plotly.graph_objects as go # Import graph_objects for annotations

# Define a color map for the 'Survived' column
color_map = {0: 'black', 1: '#57B9FF'}

# Define the mapping for Passenger Class labels
pclass_labels = {1: '👑 First Class', 2: '💵 Second Class', 3: '🪙 Third Class'}

fig = px.histogram(new_df, x='Passenger_Class', color='Survived',
                   title='Distribution of Passengers by Passenger Class and Survival',
                   barmode='group',
                   color_discrete_map=color_map,
                   width=900, height=600) # Set the chart size

# Update layout for better readability
fig.update_layout(
    xaxis_title=None, # Set x-axis title to None to hide it
    yaxis_title='Number of Passengers',
    yaxis={'visible': False, 'showticklabels': False},
    plot_bgcolor='white',
    paper_bgcolor='white',
    xaxis=dict(
        tickmode = 'array',
        tickvals = list(pclass_labels.keys()), # Use the numerical class values as tickvals
        ticktext = list(pclass_labels.values()), # Use the string labels as ticktext
        tickfont = dict(weight='bold', size=14) # Make x-axis labels bold and increase size
    ),
    shapes=[
        dict(
            type='line',
            xref='paper', x0=0, x1=1,
            yref='paper', y0=0, y1=0,
            line=dict(color='grey', width=2)
        )
    ]
)

# Add text annotations on top of the bars and style them
fig.update_traces(texttemplate='%{y}', textposition='outside')

# Update text font based on color
for i, trace in enumerate(fig.data):
    survived_status = int(trace.name) # Get the survival status from the trace name (0 or 1)
    color = color_map.get(survived_status) # Get the color from the color_map
    if color:
        fig.update_traces(selector=dict(name=str(survived_status)), # Select trace by string name
                          textfont=dict(color=color, size=14, weight='bold')) # Update text font properties

# Update legend labels for the 'Survived' column
new_legend_names = {0: '💀 No', 1: '🛟 Yes'} # Added emojis to legend
fig.for_each_trace(lambda t: t.update(name = new_legend_names[int(t.name)]))

# Add annotations for survival rate below each Passenger Class label
annotations = []
# Make sure survival_rate_table_pclass is available from previous cells
# If not, recalculate it here or ensure the cell B9Xe9SDwqfWZ is run before this one
if 'survival_rate_table_pclass' in locals():
    for pclass, row in survival_rate_table_pclass.iterrows():
        survival_rate = row['Survival Rate'] * 100 # Convert to percentage
        annotations.append(dict(
            xref='x', yref='paper', # Use 'x' for x-axis data values, 'paper' for y-axis as fraction of plot height
            x=pclass, y=-0.04, # Position below the x-axis labels (adjust y as needed)
            text=f'Survival rate: {survival_rate:.1f}%', # Format as percentage with one decimal
            showarrow=False,
            font=dict(size=12, color='grey'), # Adjust font size and color as needed
            xanchor='center', yanchor='top'
        ))
    fig.update_layout(annotations=annotations, margin=dict(b=100)) # Add annotations to layout and increase bottom margin


fig.show()

In [42]:
import pandas as pd

# Create a crosstab to get survival counts by Passenger_Class
survival_counts_pclass = pd.crosstab(new_df['Passenger_Class'], new_df['Survived'])

# Rename columns for clarity
survival_counts_pclass = survival_counts_pclass.rename(columns={0: 'Died', 1: 'Survived'})

# Calculate the total count for each Passenger_Class
survival_counts_pclass['Total'] = survival_counts_pclass['Died'] + survival_counts_pclass['Survived']

# Calculate the survival rate (Survived / Total)
survival_counts_pclass['Survival Rate'] = survival_counts_pclass['Survived'] / survival_counts_pclass['Total']

# Select and reorder columns for the final table
survival_rate_table_pclass = survival_counts_pclass[['Total', 'Survived', 'Died', 'Survival Rate']].copy()

# Format the 'Survival Rate' column as percentage with one decimal and make text bold
# Add a light blue bar chart in the background for Survival Rate column with white border
display(survival_rate_table_pclass.sort_values(by='Passenger_Class', ascending=True).style
        .bar(subset=['Survival Rate'], color='lightblue', props='border: 1px solid white;')
        .format({'Survival Rate': lambda x: f'<b>{x:.1%}</b>'})
        .set_properties(**{'text-align': 'center'})) # Center align all columns for simplicity initially

Survived,Total,Survived,Died,Survival Rate
Passenger_Class,,,,
1,323,200,123,61.9%
2,277,119,158,43.0%
3,709,181,528,25.5%


## Modeling Survival Rate